# Account & Registration — test job

Runs the **same code as the `clearing_backlog_accounts` DAG**, from this notebook, against StarRocks:

1. **Extract**: the DAG's own extract SQL (`accounts_clearing_tickets._oci_sql` / `_sprinklr_sql`)
2. **Process**: `dag_pipeline.process_tickets.process_ticket` → accounts engine → routing rules → customer reply
3. **Write**: `dag_pipeline.write_service_work.result_to_service_work_row`. **Off by default** (dry run)

Nothing is reimplemented here; if a rule or query changes in the repo, this notebook picks it up.

Routing follows the Account & Registration booklet (CIF status × Application Check matrix):

| Case | Team |
|---|---|
| Request to correct personal data | Compliance |
| ACTIVE CIF + CLOSED account | Account Operation |
| ACTIVE + checks passed (closure request → confirmation reply) | Account Operation |
| DECEASED / FINAL_EXIT / END_RELATIONSHIP + passed (balance > 0 → document request) | Account Operation (balance transfer) |
| BLOCKED + passed | Compliance |
| ID_EXPIRED + passed | Technology |
| NO CIF / UNVERIFIED / NAFATH_REQUIRED / KYC_UPDATE / RE_VERIFY + PENDING/FAILED | Compliance |
| … + INCOMPLETE (customer-completable → ask customer; internal/sensitive → neutral reply) | Technology |
| Anything else | `accounts/NO_MATCH` (human review) |

## 1. Settings: fill in the StarRocks connection

In [ ]:
# ── StarRocks connection (MySQL protocol) ── PLACEHOLDERS: fill these in ──
STARROCKS_HOST     = "<starrocks-fe-host>"      # e.g. kube-starrocks-fe-service.starrocks.svc.cluster.local
STARROCKS_PORT     = 9030                       # StarRocks FE query port
STARROCKS_USER     = "<username>"
STARROCKS_PASSWORD = "<password>"               # better: os.environ["STARROCKS_PASSWORD"]
STARROCKS_DATABASE = None                       # optional default database; queries use full names

# ── Path to the clearing-tickets repo (folder that contains config.py) ──
REPO_PATH = ".."                                # this notebook lives in <repo>/notebooks

# ── Run parameters (same as the DAG's params) ──
LIMIT            = 20        # tickets per source, newest first
LOOKBACK_DAYS    = 30        # 0 = no date filter
SOURCE           = "oci"     # "oci" / "sprinklr" / "all" (Sprinklr needs config.SPRINKLR_ACCOUNTS)
INCLUDE_RESOLVED = False     # False = skip ORA_SVC_RESOLVED tickets
SR_NUMBER        = ""        # set to test one ticket by number (bypasses the filters)
MAX_WORKERS      = 4         # tickets processed in parallel (1 = one at a time)

# ── Writing ── keep False to test without touching any table ──
WRITE_RESULTS = False

## 2. Imports (same modules the DAG uses)

In [ ]:
import os, sys, types, threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = Path(REPO_PATH).resolve()
assert (ROOT / "config.py").exists(), f"config.py not found in {ROOT}: fix REPO_PATH"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# The DAG files import Airflow. Outside Airflow, register tiny stand-ins so
# the same files can be imported. Nothing Airflow-specific is used here.
try:
    import airflow  # noqa: F401
except ImportError:
    def _mod(name, **attrs):
        m = types.ModuleType(name); m.__dict__.update(attrs); sys.modules[name] = m; return m
    class _DAG:
        def __init__(self, *a, **k): pass
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _Op:
        def __init__(self, *a, **k): pass
        def __rshift__(self, other): return other
    class _Variable:
        @staticmethod
        def get(key, default_var=None): return default_var
    _mod("airflow", DAG=_DAG)
    _mod("airflow.operators"); _mod("airflow.operators.python", PythonOperator=_Op)
    _mod("airflow.models", Variable=_Variable)
    _mod("airflow.utils"); _mod("airflow.utils.dates", days_ago=lambda n: None)

import pandas as pd
import config
import accounts_clearing_tickets as accounts_dag                      # extract SQL
from services.product_services import set_query_runner, load_sql
from services.products.accounts import AccountsEngine
from dag_pipeline.process_tickets import process_ticket               # same per-ticket entry point as the DAG
from dag_pipeline.write_service_work import result_to_service_work_row, insert_service_work_rows

pd.set_option("display.max_colwidth", 120)
print("repo:", ROOT)

## 3. StarRocks connection: one per thread, used by every query

In [ ]:
import mysql.connector  # pip install mysql-connector-python

_local = threading.local()

def _connection():
    conn = getattr(_local, "conn", None)
    if conn is None or not conn.is_connected():
        conn = mysql.connector.connect(
            host=STARROCKS_HOST, port=STARROCKS_PORT,
            user=STARROCKS_USER, password=STARROCKS_PASSWORD,
            database=STARROCKS_DATABASE, autocommit=True,
        )
        _local.conn = conn
    return conn

def run_query(query, params=None):
    """SELECT -> list of dicts. Same @placeholder replacement as the DAG."""
    for key, value in (params or {}).items():
        query = query.replace("@" + str(key), str(value))
    cur = _connection().cursor()
    try:
        cur.execute(query)
        rows = cur.fetchall() or []
        cols = [d[0] for d in cur.description] if cur.description else []
        return [dict(zip(cols, r)) for r in rows]
    finally:
        cur.close()

def execute_sql(query, many=None):
    cur = _connection().cursor()
    try:
        cur.execute(query) if many is None else cur.executemany(query, many)
        return cur.rowcount
    finally:
        cur.close()

# Engine queries (customer profile, names) go through this connection.
set_query_runner(run_query)
print(run_query("SELECT 1 AS ok"))

## 4. Extract: the DAG's own extract SQL

In [ ]:
opts = dict(limit=LIMIT, lookback_days=LOOKBACK_DAYS,
            include_resolved=INCLUDE_RESOLVED, sr_number=SR_NUMBER.strip() or None)

tickets = []
if SOURCE in ("oci", "all"):
    rows = run_query(accounts_dag._oci_sql(**opts))
    for t in rows: t["source"] = "oci"
    tickets += rows
if SOURCE in ("sprinklr", "all"):
    sprinklr_cfg = accounts_dag._sprinklr_config()   # None until config.SPRINKLR_ACCOUNTS is filled in
    if sprinklr_cfg is None:
        print("Sprinklr not configured yet (config.SPRINKLR_ACCOUNTS): skipped")
    else:
        rows = run_query(accounts_dag._sprinklr_sql(sprinklr_cfg, **opts))
        for t in rows: t["source"] = "sprinklr"
        tickets += rows
for t in tickets:
    t["product"] = accounts_dag.PRODUCT

print(f"{len(tickets)} tickets")
pd.DataFrame(tickets)[["source", "SrNumber", "CreationDate", "StatusTypeCd", "CategoryLevel2_c", "BaraqInternalID_c"]] if tickets else None

## 5. Process: same function the DAG calls for each ticket

In [ ]:
def _run_one(ticket):
    r = process_ticket(ticket)            # never raises; errors come back as ok=False
    r["_source"] = ticket.get("source")
    return r

if MAX_WORKERS > 1 and len(tickets) > 1:
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, len(tickets))) as pool:
        results = list(pool.map(_run_one, tickets))   # keeps ticket order
else:
    results = [_run_one(t) for t in tickets]

rows = []
for r in results:
    txn = r.get("transaction") or {}
    sw = result_to_service_work_row(r, mode="notebook") if (r.get("ticket") or {}).get("SrNumber") else {}
    rows.append({
        "source": r["_source"],
        "SrNumber": r["ticket"].get("SrNumber"),
        "subcategory": r["ticket"].get("CategoryLevel2_c"),
        "cif_status": (txn.get("cif_status") or "NO_CIF") if txn else None,   # blank = NO_CIF, as the rules see it
        "app_checks_not_passed": txn.get("app_checks_not_passed"),
        "incomplete": txn.get("app_checks_incomplete_list"),
        "account_status": txn.get("account_status"),
        "balance": txn.get("wallet_balance"),
        "approval_band": txn.get("approval_band"),
        "rule_id": r.get("rule_id"),
        "route": sw.get("route"),
        "ticket_status": sw.get("ticket_status"),
        "reply_en": r.get("response_en"),
        "reply_ar": r.get("response_ar"),
        "error": r.get("error"),
    })
df = pd.DataFrame(rows)
print(f"ok={sum(r['ok'] for r in results)}  errors={sum(not r['ok'] for r in results)}  "
      f"routed={df['rule_id'].notna().sum() if len(df) else 0}")
df

### Summary by route

In [ ]:
df.groupby(["route", "rule_id"], dropna=False).size().rename("tickets").reset_index() if len(df) else None

### Errors (if any)

In [ ]:
for r in results:
    if not r["ok"]:
        print(r["ticket"].get("SrNumber"), r.get("error_type"), r.get("error"))

## 6. One customer: see exactly what the engine sees

Shows the customer profile query result (CIF status, latest check per type, account, balance) and the decision for a single User ID, without needing a ticket.

In [ ]:
TEST_USER_ID = "<BaraqInternalID_c>"          # placeholder: a User ID to inspect
TEST_SUBCATEGORY = "I have an issue with my registration"

if not TEST_USER_ID.startswith("<"):
    engine = AccountsEngine()
    probe = {"SrNumber": "NOTEBOOK-TEST", "BaraqInternalID_c": TEST_USER_ID,
             "CategoryLevel1_c": "Account and Registration",
             "CategoryLevel2_c": TEST_SUBCATEGORY, "product": "accounts"}
    profile = engine.fetch_transaction(probe)
    display(pd.Series(profile or {}, name="profile (SQL)"))
    row = engine.build_row(probe, profile or {})
    display(pd.Series({k: row[k] for k in ["cif_status", "app_check_status", "incomplete_action",
                                            "account_status", "has_balance", "approval_band"]}, name="engine row"))
    decision = engine.match(row, engine.get_rules())
    display(pd.Series(decision, name="decision"))

## 7. Write (optional)

Builds the exact rows the DAG writes. **Only inserts when `WRITE_RESULTS = True`**, into `config.ACCOUNTS_TARGET_TABLE` (routed) and `config.ACCOUNTS_NO_MATCH_TABLE` (not routed).

In [ ]:
def _is_matched(r):   # same rule as the DAG: rule matched AND a reply exists
    return r.get("rule_id") is not None and bool((r.get("response_en") or "").strip() or (r.get("response_ar") or "").strip())

writable = [r for r in results if (r.get("ticket") or {}).get("SrNumber")]
matched_rows   = [result_to_service_work_row(r, mode="notebook") for r in writable if _is_matched(r)]
unmatched_rows = [result_to_service_work_row(r, mode="notebook") for r in writable if not _is_matched(r)]
print(f"would write {len(matched_rows)} -> {config.ACCOUNTS_TARGET_TABLE}")
print(f"would write {len(unmatched_rows)} -> {config.ACCOUNTS_NO_MATCH_TABLE}")

if WRITE_RESULTS:
    n1 = insert_service_work_rows(execute_sql, config.ACCOUNTS_TARGET_TABLE, matched_rows)
    n2 = insert_service_work_rows(execute_sql, config.ACCOUNTS_NO_MATCH_TABLE, unmatched_rows)
    print(f"inserted {n1} + {n2} rows")
else:
    print("dry run: set WRITE_RESULTS = True to insert")
pd.DataFrame(matched_rows + unmatched_rows)[["sr_number", "route", "ticket_status", "resolution", "facts"]] if writable else None